In [0]:
%pip install plotly

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()

league_avg = spark.table("workspace.gold_nba.pace_revolution") \
    .filter(F.col("three_pt_rate").isNotNull()) \
    .groupBy("season_year") \
    .agg(
        F.round(F.avg("pace"), 1).alias("pace"),
        F.round(F.avg("three_pt_rate"), 3).alias("three_pt_rate"),
        F.round(F.avg("avg_fg3a"), 1).alias("avg_fg3a")
    ) \
    .orderBy("season_year") \
    .toPandas()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=league_avg["season_year"], y=league_avg["pace"],
               name="Pace (possessions/game)", line=dict(color="#1d428a", width=2.5)),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(x=league_avg["season_year"], y=league_avg["three_pt_rate"],
               name="3pt Attempt Rate", line=dict(color="#c8102e", width=2.5)),
    secondary_y=True
)


annotations = [
    (1994, "3pt line shortened"),
    (1997, "3pt line restored"),
    (2013, "Warriors dynasty begins"),
    (2016, "73-win season"),
]

for year, label in annotations:
    fig.add_vline(x=year, line_dash="dash", line_color="gray", opacity=0.5)
    fig.add_annotation(x=year, y=1.05, text=label, showarrow=False,
                       xref="x", yref="paper", textangle=-45, font=dict(size=9))

fig.update_layout(
    title="The NBA Pace Revolution (1985-2022)",
    xaxis_title="Season",
    height=500,
    legend=dict(x=0.01, y=0.99),
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_yaxes(title_text="Pace (possessions/game)", secondary_y=False)
fig.update_yaxes(title_text="3pt Attempt Rate", secondary_y=True)

fig.show()